# Use case: an agent that remembers across runs

A coding agent learns things during a task: which test was flaky and why, which command to
run, a decision the user made. Next week, in a fresh process, it has forgotten all of it.

Two stores fix that. A **session** holds the conversation as turns, searchable and replayable.
A **facts** topic holds durable notes the agent chose to keep. Before every prompt the agent
pulls the relevant facts and the relevant earlier turns into its context. This notebook builds
that loop, runs a few turns, then *closes everything and reopens it* to show the memory survives.

**Needs:** Ollama with `nomic-embed-text`. The turns that call the model need `qwen2.5:7b-instruct`.

In [1]:
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
STORE = ROOT / ".usecase_nb" / "agent"          # the store lives here; delete the folder to start over
RUN_LLM = True                               # the steps that call a chat model are slow on CPU
LLM = "qwen2.5:7b-instruct"                      # follows context better than llama3.2:3b

from slim_llm_memory import topic, session
from slim_llm_memory.llm import chat

facts = topic("agent facts", path=STORE / "facts")
chat_log = session("2026-09-14 pipeline", path=STORE / "session")

## 1. What the agent learned last week

Pretend a previous run stored these. Each fact is one short doc with a name, so it can be
updated by re-adding under the same name, or removed with `forget()`.

In [2]:
facts.add({
    "flaky-test":   "test_ingest_parallel is flaky because it shares one tmp dir across workers. "
                    "Use tmp_path per test; do not mark it xfail.",
    "run-tests":    "Run the test suite with `make test`, not `pytest` directly: make sets DATABASE_URL to the "
                    "local docker Postgres on port 5433.",
    "lint-rule":    "The user wants ruff to stay at line length 100. Do not reformat to 88.",
    "deploy":       "Deploys go through `make release TAG=x.y.z`; it pushes the tag and CI does the rest. "
                    "Never push to main directly.",
    "owner-alerts": "Alerting is owned by the platform team. Changes to alert rules need a review from them.",
})

added 5 doc(s), 5 chunks: 5 embedded, 0 unchanged, 0 removed

## 2. The loop

`remember()` builds the context: the best facts for this prompt, then the most relevant earlier
turns, then the last few turns in order. `step()` sends that to the model and records both sides
of the exchange in the session.

In [3]:
def remember(prompt, k_facts=3, k_recall=3, last=4):
    parts = []
    f = facts.ask(prompt, k=k_facts, min_score=0.35)
    if f:
        parts.append("Facts from earlier sessions:\n" + "\n".join(f"- {h.text}" for h in f))
    if len(chat_log):
        rc = chat_log.recall(prompt, k=k_recall, min_score=0.35)
        if rc:
            parts.append("Relevant earlier turns:\n" + "\n".join(f"- {h.meta['role']}: {h.text}" for h in rc))
        parts.append("Last turns:\n" + "\n".join(f"{r}: {t}" for r, t in chat_log.history(last)))
    return "\n\n".join(parts)

def step(prompt):
    ctx = remember(prompt)
    chat_log.turn("user", prompt)
    if not RUN_LLM:
        print("[context the model would see]\n" + ctx); return
    reply = chat(LLM, [
        {"role": "system", "content": "You are a coding assistant. Use the facts and history given. "
                                      "Answer in at most three sentences."},
        {"role": "user", "content": f"{ctx}\n\nUser: {prompt}"},
    ], options={"num_predict": 120})
    chat_log.turn("assistant", reply)
    print("USER:     ", prompt)
    print("CONTEXT:  ", ctx.count("\n") + 1, "lines,", len(ctx.split()), "words")
    print("ASSISTANT:", reply, "\n")

In [4]:
step("the parallel ingest test just failed again, should I mark it xfail?")
step("ok, how do I run the whole suite properly here?")

USER:      the parallel ingest test just failed again, should I mark it xfail?
CONTEXT:   4 lines, 64 words
ASSISTANT: No, you should not mark it xfail. Instead, ensure each test uses `tmp_path` to avoid sharing temporary files across workers. This should resolve the issue. 



USER:      ok, how do I run the whole suite properly here?
CONTEXT:   12 lines, 153 words
ASSISTANT: To run the entire test suite properly, use `make test`. This command sets up the local Docker Postgres database on port 5433 as specified by `DATABASE_URL`. Ensure all tests, including the parallel ingest test, are configured to use `tmp_path` to avoid sharing temporary files across workers. 



## 3. Restart

Close both stores, open them again, and ask something that only makes sense with the earlier
conversation. The session turns and the facts come back from disk.

In [5]:
facts.close(); chat_log.close()

facts = topic("agent facts", path=STORE / "facts")
chat_log = session("2026-09-14 pipeline", path=STORE / "session")
print(chat_log, "\n")
step("remind me what we said about that test, and then tell me how a release is cut")

session('2026-09-14 pipeline': 4 turns, /home/trbck/workspace/slim-llm-memory/.usecase_nb/agent/session) 



USER:      remind me what we said about that test, and then tell me how a release is cut
CONTEXT:   15 lines, 259 words
ASSISTANT: The parallel ingest test should not be marked xfail; instead, ensure each test uses `tmp_path` to avoid sharing temporary files. A release is cut by running `make release TAG=x.y.z`, which pushes the tag, and CI handles the rest without directly pushing to main. 



## 4. The agent decides to remember something new

A fact learned in this run goes into `facts` under a name. Re-adding under the same name later
replaces it, so the store holds the current truth, not a history.

In [6]:
facts.add("Port 5433 is taken by another project on this machine; the local Postgres for this repo "
          "now runs on 5434 and make test was updated to match.", name="run-tests")
print(facts.ask("which port is the test database on?", k=1).top.text)

Port 5433 is taken by another project on this machine; the local Postgres for this repo now runs on 5434 and make test was updated to match.


## 5. Summarise the session for a hand-off

`summary()` is one model call over the transcript. Store the result as a fact and the next
session starts with it. Note that the summary still mentions port 5433: it records what was
*said*, not what is true now. The named fact `run-tests` holds the current truth; the summary
is history. Both are useful, and naming keeps them apart.

In [7]:
if RUN_LLM:
    s = chat_log.summary(model=LLM)
    print(s)
    facts.add(s, name="summary-2026-09-14")

- The parallel ingest test failure should not be marked xfail; use `tmp_path` for each test to avoid shared temporary files.
- Run the entire test suite properly using `make test`, which sets up a local Docker Postgres database on port 5433.
- Ensure tests are configured with `tmp_path` to prevent sharing of temporary files across workers.
- For releasing, use `make release TAG=x.y.z` to push the tag; CI handles the rest without directly pushing to main.


## Takeaways

- **Two stores, two lifetimes.** The session is the transcript; facts are what survives the task.
- **Retrieve before every prompt.** Three facts and three turns cost one embedding call and add a few hundred words.
- **Name facts.** `add(text, name=...)` makes memory updatable instead of append-only.
- **Restart is free.** Everything is on disk after every call; there is nothing to save or load.

In [8]:
facts.close(); chat_log.close()